# L2 step 4c · Every 7B and 3B on the benchmark's six tasks, side by side (your PC, no GPU)

Abrar's request (23 Nov): a detailed comparison, task by task, of the base 7B, DisasterM3's own fine-tuned 7B, our fine-tuned 7B, and our two distilled 3Bs. No model runs here: every number comes from answers already saved, or from the DisasterM3 paper.

**The models**

| Model | What it is | Where its numbers come from |
|---|---|---|
| Qwen2.5-VL-3B, untrained | the 3B both of ours started from | the DisasterM3 paper, Table 2 |
| our 3B (b), soft labels | trained to match the 7B's 20 most likely words at each step | L2 run 5, rerun with the overflow fix (23 Nov) |
| our 3B (c), the 7B's answers | trained on the 7B's checked answers, counting and relational questions answered again with the options shown; **the one kept** | L2 run 5, rerun with the fix |
| our 3B (c) in 4 bits | the same, in 4 bits (bitsandbytes NF4): the field model | L3, rerun with the fix |
| Qwen2.5-VL-7B, untrained | the 7B ours started from | the paper, Table 2 |
| our first 7B | our CSE499A fine-tune | L0 step 2, re-scored at full size |
| our 7B | L0 step 2's fine-tune, the 3Bs' teacher | L0 step 2 |
| DisasterM3's fine-tuned 7B | the paper's authors' fine-tune of the same model on the same training set | the paper, Table 2 |

**The tasks**
- **DTR:** disaster type.
- **BBR:** bearing bodies, which kinds of objects were damaged.
- **BDC:** building damage counting.
- **DRE:** road damage, the share of damaged road.
- **DSR:** disaster scene, land-use types.
- **ORR:** relational reasoning, how two objects relate.

The mean is the plain mean of the six, as the paper's average.

**What's comparable, and what isn't**
- **Same questions:** the benchmark's 10,675 optical questions (the paper's "optical-optical" setting).
- **Decoding:** ours is greedy; the paper's is vLLM's defaults, effectively greedy (H1, 19 Oct).
- **Image size:** our 7Bs read full size (1036 px). Our 3Bs read their training size (at most 768 × 28 × 28 pixels, 756 px for a 1024 px photo). The paper's 3B reads full size.
- **Clean scores:** E5's clean scores (the 186 questions whose photos are also training photos left out) exist only for our models, since the paper published no per-question answers.
- **Counting accuracy:** on these options it can't show counting on its own (a rule that never sees the photo scores 67.0% on building counting). So counting is also shown on the "fair" questions, whose largest option is 20 or less (L2 step 4b).
- **Paired tests:** our models answered the same questions, so they're compared question by question (exact McNemar). The paper's models can't be.

**How to run:** on the PC, in Jupyter (cse499 environment), from `Champion_Task/`, after run 5's and L3's fixed full runs are saved in `L2-step4-results/full/` and `L3-results/full/`. Run all; it takes seconds. Writes `L2-step4-results/full/comparison/`.

In [1]:
# ── Cell 1: The saved answers, and the paper's Table 2 ─────────────────────────────
import ast
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

CT = Path.cwd()                                            # Champion_Task
RUN5 = CT / "L2-step4-results" / "full" / "L2S4"           # the 3Bs (c) and (b), normal precision, with the overflow fix
L3 = CT / "L3-results" / "full" / "L3"                     # the 3B (c) in 4 bits, with the fix
HELD_OUT = CT / "L2-step3-results" / "full" / "L2S3" / "per_question.csv"
SEVEN_B = {k: [CT / "L0-step2-results" / f"{k}-part{p}" / "L0S2" / f"L0_step2_{k}_per_question.csv" for p in "AB"]
           for k in ("old", "new")}
FIRST_7B_DTR_ORR = CT / "L0-step1-results" / "L0S1" / "L0_step1_per_question.csv"   # its full-size DTR and ORR (L0 step 1)
FLAGGED = CT / "E5-results" / "step1" / "bench_questions_flagged.csv"
OUT = CT / "L2-step4-results" / "full" / "comparison"
OUT.mkdir(parents=True, exist_ok=True)
TASKS = ["DTR", "BBR", "BDC", "DRE", "DSR", "ORR"]
NAMES = {"DTR": "disaster type", "BBR": "bearing bodies", "BDC": "building counting", "DRE": "road damage",
         "DSR": "scene (land use)", "ORR": "relational reasoning"}
# The DisasterM3 paper (NeurIPS 2025 Datasets and Benchmarks), Table 2, Bench set, optical-optical setting
PAPER = {"Qwen2.5-VL-3B, untrained (paper)": {"DSR": 30.8, "DTR": 56.1, "BBR": 5.7, "BDC": 29.9, "DRE": 21.2, "ORR": 13.8},
         "Qwen2.5-VL-7B, untrained (paper)": {"DSR": 28.3, "DTR": 66.6, "BBR": 4.7, "BDC": 34.2, "DRE": 29.3, "ORR": 23.9},
         "DisasterM3's fine-tuned 7B (paper)": {"DSR": 37.7, "DTR": 83.6, "BBR": 21.5, "BDC": 34.3, "DRE": 29.4, "ORR": 36.2}}
PAPER_AVG = {"Qwen2.5-VL-3B, untrained (paper)": 26.2, "Qwen2.5-VL-7B, untrained (paper)": 31.2,
             "DisasterM3's fine-tuned 7B (paper)": 40.4}
SCALE = {"Qwen2.5-VL-72B, untrained (paper)": {"DSR": 47.0, "DTR": 74.8, "BBR": 6.8, "BDC": 34.8, "DRE": 28.9, "ORR": 50.8},
         "GPT-4.1 (paper)": {"DSR": 52.4, "DTR": 79.6, "BBR": 7.2, "BDC": 25.5, "DRE": 25.0, "ORR": 64.0},
         "InternVL3-8B fine-tuned (paper)": {"DSR": 42.6, "DTR": 79.3, "BBR": 23.9, "BDC": 29.1, "DRE": 24.9, "ORR": 50.6}}
for name, avg in PAPER_AVG.items():                      # the paper's average is the plain mean of the six tasks
    assert abs(np.mean(list(PAPER[name].values())) - avg) < 0.06, name

for d, models in ((RUN5, {"b", "c"}), (L3, {"c"})):
    cfg = json.loads((d / "config.json").read_text(encoding="utf-8"))
    assert not cfg["smoke_test"] and cfg["complete"] and "fix" in cfg, f"{d}: not a complete run with the overflow fix"
    assert models <= set(cfg.get("models", [cfg["winner"]])), f"{d}: missing {models}"

def letters(text):
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

q = pd.read_csv(RUN5 / "per_question.csv")
seven = {k: pd.concat([pd.read_csv(p) for p in v]).dropna(subset=["answer"]).drop_duplicates("id").set_index("id")
         for k, v in SEVEN_B.items()}
nf4 = pd.read_csv(L3 / "per_question.csv", usecols=["id", "answer", "correct"]).set_index("id")
# Our first 7B at full size: L0 step 2's re-score has its BBR, BDC, DRE and DSR; L0 step 1 its DTR and ORR
first_dtr_orr = pd.read_csv(FIRST_7B_DTR_ORR).set_index("id")["full_correct"]
first_7b = q["id"].map(seven["old"]["correct"]).combine_first(q["id"].map(first_dtr_orr))
OURS = {"our 3B (b), soft labels": q["correct_b"].astype(bool),
        "our 3B (c), the 7B's answers": q["correct_c"].astype(bool),
        "our 3B (c) in 4 bits": q["id"].map(nf4["correct"]).astype(bool),
        "our first 7B (CSE499A)": first_7b.fillna(False).astype(bool),
        "our 7B (L0 step 2)": q["id"].map(seven["new"]["correct"]).fillna(False).astype(bool)}
for name, ok in OURS.items():
    q[name] = ok.to_numpy()
flag = pd.read_csv(FLAGGED)
copied = set(flag.loc[(flag["sensor"] == "Optical") & (flag["flag"] == "after-image copied"), "id"])
q["clean"] = ~q["id"].isin(copied)
assert len(q) == 10675 and q["id"].isin(copied).sum() == 186
PUBLISHED = {"our 7B (L0 step 2)": {"DTR": 76.77, "BBR": 19.99, "BDC": 30.31, "DRE": 33.03, "DSR": 40.46, "ORR": 34.62},
             "our first 7B (CSE499A)": {"DTR": 75.64, "BBR": 20.94, "BDC": 30.09, "DRE": 32.39, "DSR": 40.06, "ORR": 31.12}}
for name, ref in PUBLISHED.items():                      # L0 steps 1 and 2's published scores, reproduced
    for t, v in ref.items():
        got = 100 * q.loc[q["task"] == t, name].mean()
        assert abs(got - v) < 0.011, f"{name} {t}: {got:.2f} against {v}"
print(f"✓ {len(q):,} questions; {len(OURS)} of our models; the paper's three averages reproduced from its task scores")

✓ 10,675 questions; 5 of our models; the paper's three averages reproduced from its task scores


In [2]:
# ── Cell 2: The six tasks, every model ──────────────────────────────────────────
ORDER = ["Qwen2.5-VL-3B, untrained (paper)", "our 3B (b), soft labels", "our 3B (c), the 7B's answers", "our 3B (c) in 4 bits",
         "Qwen2.5-VL-7B, untrained (paper)", "our first 7B (CSE499A)", "our 7B (L0 step 2)", "DisasterM3's fine-tuned 7B (paper)"]

def scores(subset=None):
    d = q if subset is None else q[subset]
    rows = {}
    for name in ORDER:
        if name in PAPER:
            r = dict(PAPER[name]) if subset is None else {t: float("nan") for t in TASKS}
        else:
            r = {t: 100 * d.loc[d["task"] == t, name].mean() for t in TASKS}
        rows[name] = {**r, "mean": float(np.mean([r[t] for t in TASKS]))}
    return pd.DataFrame(rows).T[TASKS + ["mean"]].round(2)

ALL = scores()
CLEAN = scores(q["clean"])
gains = pd.DataFrame({
    "our 3B (c) − untrained 3B": ALL.loc["our 3B (c), the 7B's answers"] - ALL.loc["Qwen2.5-VL-3B, untrained (paper)"],
    "our 3B (c) − our 3B (b)": ALL.loc["our 3B (c), the 7B's answers"] - ALL.loc["our 3B (b), soft labels"],
    "our 3B (c) − our 7B (its teacher)": ALL.loc["our 3B (c), the 7B's answers"] - ALL.loc["our 7B (L0 step 2)"],
    "4 bits − normal precision, (c)": ALL.loc["our 3B (c) in 4 bits"] - ALL.loc["our 3B (c), the 7B's answers"],
    "our 7B − untrained 7B": ALL.loc["our 7B (L0 step 2)"] - ALL.loc["Qwen2.5-VL-7B, untrained (paper)"],
    "our 7B − DisasterM3's 7B": ALL.loc["our 7B (L0 step 2)"] - ALL.loc["DisasterM3's fine-tuned 7B (paper)"],
}).T.round(2)
print(ALL.to_string()); print(); print(gains.to_string())

                                      DTR    BBR    BDC    DRE    DSR    ORR   mean
Qwen2.5-VL-3B, untrained (paper)    56.10   5.70  29.90  21.20  30.80  13.80  26.25
our 3B (b), soft labels             63.74  21.98  19.93  27.57  36.92  24.71  32.47
our 3B (c), the 7B's answers        62.04  16.65  49.79  46.63  35.38  51.98  43.74
our 3B (c) in 4 bits                63.17  16.55  48.17  48.72  34.63  55.71  44.49
Qwen2.5-VL-7B, untrained (paper)    66.60   4.70  34.20  29.30  28.30  23.90  31.17
our first 7B (CSE499A)              75.64  20.94  30.09  32.39  40.06  31.12  38.37
our 7B (L0 step 2)                  76.77  19.99  30.31  33.03  40.46  34.62  39.20
DisasterM3's fine-tuned 7B (paper)  83.60  21.50  34.30  29.40  37.70  36.20  40.45

                                     DTR    BBR    BDC    DRE    DSR    ORR   mean
our 3B (c) − untrained 3B           5.94  10.95  19.89  25.43   4.58  38.18  17.49
our 3B (c) − our 3B (b)            -1.70  -5.33  29.86  19.06  -1.54  27.27  

In [3]:
# ── Cell 3: Question by question among our models (exact McNemar), the clean scores, and fair counting ──
def mcnemar_exact(only_a, only_b):
    n = only_a + only_b
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(only_a, only_b) + 1)) / 2 ** n)

assert abs(mcnemar_exact(0, 5) - 0.0625) < 1e-12
PAIRS = [("our 3B (c), the 7B's answers", "our 3B (b), soft labels"), ("our 3B (c), the 7B's answers", "our 7B (L0 step 2)"),
         ("our 3B (c) in 4 bits", "our 3B (c), the 7B's answers"), ("our 7B (L0 step 2)", "our first 7B (CSE499A)")]
tests = []
for a, b in PAIRS:
    for t in TASKS + ["all six"]:
        d = q if t == "all six" else q[q["task"] == t]
        only_a, only_b = int((d[a] & ~d[b]).sum()), int((~d[a] & d[b]).sum())
        tests.append({"A": a, "B": b, "task": t, "questions": len(d), "only_A_right": only_a, "only_B_right": only_b,
                      "A_minus_B_points": round(100 * (only_a - only_b) / len(d), 2), "p": mcnemar_exact(only_a, only_b)})
tests = pd.DataFrame(tests)
tests["beyond_chance"] = tests["p"] < 0.05

def option_values(v):
    return [float(x) for x in (ast.literal_eval(v) if isinstance(v, str) else v)]

def blind_rule(values):
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

cnt = q[q["task"].isin(["BDC", "DRE"])].copy()
cnt["vals"] = cnt["values"].map(option_values)
cnt["fair"] = cnt["vals"].map(max) <= 20
cnt["blind rule"] = [blind_rule(v) == "".join(sorted(letters(t))) for v, t in zip(cnt["vals"], cnt["truth"])]
fair = pd.DataFrame({name: {t: round(100 * cnt.loc[(cnt["task"] == t) & cnt["fair"], name].mean(), 2) for t in ("BDC", "DRE")}
                     for name in list(OURS) + ["blind rule"]}).T
fair["questions"] = [int(((cnt["task"] == "BDC") & cnt["fair"]).sum())] * len(fair)
print(tests[tests["task"] == "all six"].to_string(index=False)); print(); print(fair.to_string())

                           A                            B    task  questions  only_A_right  only_B_right  A_minus_B_points             p  beyond_chance
our 3B (c), the 7B's answers      our 3B (b), soft labels all six      10675          2428           912             14.20 4.669540e-157           True
our 3B (c), the 7B's answers           our 7B (L0 step 2) all six      10675          1732           852              8.24  2.627364e-68           True
        our 3B (c) in 4 bits our 3B (c), the 7B's answers all six      10675           439           439              0.00  1.000000e+00          False
          our 7B (L0 step 2)       our first 7B (CSE499A) all six      10675           158           115              0.40  1.088940e-02           True

                                BDC    DRE  questions
our 3B (b), soft labels       32.22  31.41       2002
our 3B (c), the 7B's answers  57.39  48.14       2002
our 3B (c) in 4 bits          53.85  51.15       2002
our first 7B (CSE499A) 

In [4]:
# ── Cell 4: Run 4's held-out questions (the choice), and the files ───────────────
held = pd.read_csv(HELD_OUT)
BANG = re.compile(r"!+")
held_rows = {}
for w, name in (("base", "Qwen2.5-VL-3B, untrained (run 4)"), ("b", "our 3B (b), soft labels"), ("c", "our 3B (c), the 7B's answers")):
    ok = held[f"{w} | correct"].astype(bool)
    held_rows[name] = {**{t: round(100 * ok[held["task"] == t].mean(), 2) for t in TASKS},
                       "rule's score": round(100 * ok[held["primary"].astype(bool)].mean(), 2),
                       "overflowed": int(held[f"{w} | answer"].astype(str).map(lambda a: bool(BANG.fullmatch(a.strip()))).sum())}
HELD = pd.DataFrame(held_rows).T

def md(df, index=True):
    try:
        return df.to_markdown(index=index)
    except ImportError:              # tabulate isn't in the cse499 environment
        return "```\n" + df.to_string(index=index) + "\n```"

ALL.to_csv(OUT / "six_tasks_all.csv")
CLEAN.to_csv(OUT / "six_tasks_clean.csv")
tests.to_csv(OUT / "paired_tests.csv", index=False)
fair.to_csv(OUT / "fair_counting.csv")
HELD.to_csv(OUT / "held_out_run4.csv")
show = ALL.rename(columns=NAMES)
summary_tests = tests[tests["task"] == "all six"][["A", "B", "only_A_right", "only_B_right", "A_minus_B_points", "p"]]
by_task = tests.pivot(index=["A", "B"], columns="task", values="A_minus_B_points")[TASKS + ["all six"]]
by_task_p = tests.pivot(index=["A", "B"], columns="task", values="beyond_chance")[TASKS + ["all six"]]
(OUT / "README.md").write_text("\n".join([
    "# Every 7B and 3B on the benchmark's six tasks", "",
    "From `l2-step4c-model-comparison.ipynb` on the PC (23 Nov), from saved answers and the DisasterM3 paper's Table 2. "
    "What each model is, and what is and isn't comparable: the notebook's first cell.", "",
    "## Accuracy (%) on the 10,675 optical questions", "", md(show), "",
    "Our 3Bs read photos at their training size (756 px); every other model at full size. Paper rows are the paper's own "
    "numbers; ours are greedy. The mean is the plain mean of the six tasks, as the paper's.", "",
    "## The differences (points)", "", md(gains.rename(columns=NAMES)), "",
    "## Our models question by question (exact McNemar)", "",
    "A − B in points over all six tasks' questions, with the questions only one of them got right:", "",
    md(summary_tests, index=False), "",
    "By task (points; a difference beyond chance, p < 0.05, is marked in `paired_tests.csv`):", "", md(by_task), "",
    "## Clean scores (without the 186 questions whose photos are training photos)", "",
    md(CLEAN.drop(index=[n for n in CLEAN.index if n in PAPER]).rename(columns=NAMES)), "",
    "## Counting on the fair questions (largest option 20 or less)", "",
    "Where the middle option isn't always right. The blind rule never sees the photo.", "", md(fair), "",
    "## For scale (the paper's Table 2)", "",
    md(pd.DataFrame(SCALE).T[TASKS].assign(mean=lambda d: d.mean(axis=1).round(2)).rename(columns=NAMES)), "",
    "## The choice between the 3Bs (run 4, 1,962 held-out training questions)", "",
    md(HELD.rename(columns=NAMES)), "",
    "The rule's score leaves out counting questions whose largest option is above 20. Each 3B lost the same 50 questions "
    "to the overflow (the vision part they share); they count as wrong for all three, so the choice stands.", "",
    "## Files", "", "`six_tasks_all.csv`, `six_tasks_clean.csv`, `paired_tests.csv`, `fair_counting.csv`, `held_out_run4.csv`",
]) + "\n", encoding="utf-8")
print(HELD.to_string()); print(f"\n✓ Wrote {OUT}")

                                    DTR    BBR    BDC    DRE    DSR   ORR  rule's score  overflowed
Qwen2.5-VL-3B, untrained (run 4)  66.67   6.75  30.94  24.24  38.79   8.0         27.80        50.0
our 3B (b), soft labels           77.38  24.94  21.99  31.96  50.57  10.0         35.95        50.0
our 3B (c), the 7B's answers      73.81  19.22  53.23  48.48  41.95  47.0         45.12        50.0

✓ Wrote d:\CSE499AB_project\Essential_task_delegation\Champion_Task\L2-step4-results\full\comparison
